# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM_AI_Agents_Bootcamp/blob/main/demo/DAY_1_DEMO_SESSION_3_HITL.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>



# About this notebook

This notebook is a hands-on tour of Human-in-the-Loop patterns for LangGraph. It shows how to add precise human control to agent workflows, from lightweight approval gates to full interactive editing, while keeping the code compact and production minded.

## What you will learn

1. How to wire LangGraph interrupts to pause a run, collect human input, and resume deterministically.
2. How to checkpoint state with `InMemorySaver` so a run can stop and continue without losing context.
3. How to wrap tools so a human can accept, edit, or override a call before it executes.
4. How to drive simple ReAct style loops that honor human review of tool calls.
5. How to implement parallel interrupts and resume them with a single map.

## Models and configuration

The notebook uses a provider switch set at startup.

* `LLM_PROVIDER=openai` uses OpenAI via `langchain_openai`.
* `LLM_PROVIDER=openrouter` uses OpenRouter via `ChatOpenAI` and a custom `base_url`.

Environment variables are loaded from `.env` via `python-dotenv`:

* `LLM_PROVIDER` (`openai` or `openrouter`)
* `OPENAI_API_KEY` when provider is OpenAI
* `OPENROUTER_API_KEY` when provider is OpenRouter
* Optional model overrides:
  * `OPENAI_MODEL` (default `gpt-4o-mini`)
  * `OPENROUTER_MODEL` (default `openai/gpt-4o-mini`)

## The patterns showcased

**Pattern A — Human feedback loop on content generation**
A simple write–review loop for LinkedIn posts. The model drafts. The human provides iterative feedback via `interrupt`. The graph loops until the human types done. Useful for any short form content workflow that benefits from iterative refinement.

**Pattern B — HITL checkpoint before a sensitive call**
The model proposes an HTTP request as JSON. A human can approve, revise, request more context, or reject before the code performs the external call. This is a minimal but powerful safety interlock for network or finance critical actions.

**Pattern C — Review and edit state**
The model writes a short summary. The human edits the text in place. The edited text becomes the new state. This pattern is ideal for compliance or brand voice checks.

**Pattern D — Parallel interrupts with a single resume map**
Two independent interrupts fire at once. The runner prints both payloads and collects a resume value for each, then resumes the graph in one step. This is a template for multi item review tasks.

**Pattern E — Tool call review inside a tiny ReAct loop**
A tool is wrapped with `add_hitl`. Before execution, a human can accept, edit arguments, or respond with a stub result. The loop continues until the model no longer requests tools. This is the smallest possible example of supervised tool use.

**Pattern F — Static interrupts for debugging**
Graph level interrupts are registered before and after specific nodes to create deterministic breakpoints. This is useful for stepwise debugging and unit style tests.

## How the interactive runner works

* Each demo builds a compiled graph and starts a new thread id for clean state.
* When an interrupt occurs, the terminal prints the payload and waits for input.
* You can paste raw strings or JSON. For the tool wrapper in Pattern E you can provide:

  * `{"type": "accept"}`
  * `{"type": "edit", "args": {"args": {"query": "weather in Zurich"}}}`
  * `{"type": "response", "args": "Skip for now"}`

For parallel interrupts the runner prints a numbered list and collects a resume value per interrupt id, then resumes once with a single `Command(resume=...)`.

## Dependencies

* `langgraph` for graphs, interrupts, checkpoints, and tasks
* `langchain_openai` for LLM bindings
* `python-dotenv` for environment loading
* `requests` used in Pattern B for a real HTTP GET

## How to run

1. Create a `.env` file with your keys and provider choice. Set `LLM_PROVIDER` to `openai` or `openrouter`, then provide the matching API key (`OPENAI_API_KEY` or `OPENROUTER_API_KEY`).
2. Run the notebook and choose a demo from the menu printed by `main()`.
3. Follow the terminal prompts to provide feedback or approvals.

## Why this matters

Real systems in finance, research, and operations often need both autonomy and control. These patterns show how to add precise human control without fighting the agent architecture. Each pattern composes cleanly, so you can start small, measure impact, and expand to richer supervision where it adds the most value.


In [1]:
!pip install -q langgraph==0.6.7 langchain-openai==0.3.33 langchain==0.3.27 python-dotenv==1.1.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.6 MB/s eta 0:00:00


In [2]:
# --- Provider + API key setup ---
# Option 1 (preferred): create a `.env` file in your project folder with e.g.
# LLM_PROVIDER=openai
# OPENAI_API_KEY=your_openai_key_here
# OPENAI_MODEL=gpt-5.4-nano-2026-03-17
#
# or:
# LLM_PROVIDER=openrouter
# OPENROUTER_API_KEY=your_openrouter_key_here
# OPENROUTER_MODEL=openai/gpt-5.4-nano7
#
# Option 2: set directly in the notebook with `%env`.

from dotenv import load_dotenv
import os

# Load from .env if available
load_dotenv()

PROVIDER = os.getenv("LLM_PROVIDER", "openai").strip().lower()
if PROVIDER not in {"openai", "openrouter"}:
    raise ValueError("LLM_PROVIDER must be 'openai' or 'openrouter'")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano-2026-03-17")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-5.4-nano")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# Fallback: ask only for the selected provider key
if PROVIDER == "openai" and not OPENAI_API_KEY:
    print("⚠️ OPENAI_API_KEY not found. Set it with `%env` or enter it below.")
    OPENAI_API_KEY = input("Enter your OPENAI_API_KEY: ").strip()

if PROVIDER == "openrouter" and not OPENROUTER_API_KEY:
    print("⚠️ OPENROUTER_API_KEY not found. Set it with `%env` or enter it below.")
    OPENROUTER_API_KEY = input("Enter your OPENROUTER_API_KEY: ").strip()

selected_model = OPENAI_MODEL if PROVIDER == "openai" else OPENROUTER_MODEL
print(f"✅ Provider: {PROVIDER} | Model: {selected_model}")



✅ Provider: openai | Model: gpt-5.4-nano-2026-03-17


# Imports

In [3]:
from __future__ import annotations
import os, uuid, json, re, sys
from typing import Any, Dict, List, Optional, TypedDict, Literal

from dotenv import load_dotenv

from langgraph.graph import StateGraph
from langgraph.constants import START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.func import entrypoint, task
from langgraph.graph.message import add_messages

from langchain_openai import ChatOpenAI
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool, BaseTool


/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


# LLM setup (OpenAI or OpenRouter via provider switch)

In [4]:
# Compatibility guard for environments that have newer `langchain` preinstalled.
# Some mixed-version setups are missing globals expected by langchain-core 0.3.x.
try:
    import langchain
    if not hasattr(langchain, "verbose"):
        langchain.verbose = False
    if not hasattr(langchain, "debug"):
        langchain.debug = False
    if not hasattr(langchain, "llm_cache"):
        langchain.llm_cache = None
except Exception:
    pass

if PROVIDER == "openrouter":
    LLM = ChatOpenAI(
        model=OPENROUTER_MODEL,
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        temperature=0,
    )
    print(f"Using OpenRouter model: {OPENROUTER_MODEL}")
else:
    LLM = ChatOpenAI(
        model=OPENAI_MODEL,
        api_key=OPENAI_API_KEY,
        temperature=0,
    )
    print(f"Using OpenAI model: {OPENAI_MODEL}")

CHECKPOINTER = InMemorySaver()

def jdump(x):
    try:
        return json.dumps(x, indent=2, ensure_ascii=False, default=str)
    except Exception:
        return str(x)



Using OpenAI model: gpt-5.4-nano-2026-03-17


# Pattern A: Human feedback loop on content generation

In [5]:
class AState(TypedDict, total=False):
    linkedin_topic: str
    generated_post: str
    human_feedback: List[str]

def a_model(state: AState) -> AState:
    topic = state["linkedin_topic"]
    fb = state.get("human_feedback", [])
    prompt = f"""
LinkedIn Topic: {topic}
Most recent human feedback: {fb[-1] if fb else "No feedback yet"}

Write a concise LinkedIn post. Consider feedback if present.
"""
    resp = LLM.invoke(prompt).content
    print("\n[model] Draft:\n" + resp + "\n")
    return {"generated_post": resp, "human_feedback": fb}

def a_human(state: AState):
    print("\n[human] awaiting feedback. Type done to finish")
    payload = {
        "generated_post": state["generated_post"],
        "message": "Provide feedback or type done"
    }
    feedback = interrupt(payload)
    print("[human] feedback:", feedback)
    if isinstance(feedback, str) and feedback.strip().lower() in {"done", "quit", "exit"}:
        return Command(goto="a_end", update={"human_feedback": state.get("human_feedback", []) + ["Finalised"]})
    return Command(goto="a_model", update={"human_feedback": state.get("human_feedback", []) + [str(feedback)]})

def a_end(state: AState) -> AState:
    print("\n[end] Final post:\n" + state["generated_post"])
    print("[end] Feedback trail:", state.get("human_feedback", []))
    return state

def build_graph_A():
    g = StateGraph(AState)
    g.add_node("a_model", a_model)
    g.add_node("a_human", a_human)
    g.add_node("a_end", a_end)
    g.set_entry_point("a_model")
    g.add_edge("a_model", "a_human")
    g.add_edge("a_end", END)
    return g.compile(checkpointer=CHECKPOINTER)


# Pattern B: HITL checkpoint before sensitive call

In [6]:
class BState(TypedDict, total=False):
    proposed_request: Dict[str, Any]
    api_result: Dict[str, Any]
    decision: str
    human_note: str

def b_propose(state: BState) -> BState:
    prompt = "Return only JSON with keys url and params for GET to https://httpbin.org/get using q and limit."
    text = LLM.invoke(prompt).content
    m = re.search(r"\{.*\}", text, re.S)
    data = {"url": "https://httpbin.org/get", "params": {"q": "fallback", "limit": 1}}
    if m:
        try:
            data = json.loads(m.group(0))
        except Exception:
            pass
    return {"proposed_request": data}

def _merge_request(current: Dict[str, Any], update: Dict[str, Any]) -> Dict[str, Any]:
    new_req = dict(current)
    if "url" in update:
        new_req["url"] = update["url"]
    if isinstance(update.get("params"), dict):
        merged_params = dict(new_req.get("params") or {})
        merged_params.update(update["params"])
        new_req["params"] = merged_params
    return new_req

def b_gate(state: BState) -> Command[Literal["b_call", "b_gate", "b_rejected"]]:
    v = interrupt({
        "question": "Review sensitive request: approve, revise, ask for more human input, or reject",
        "proposed_request": state["proposed_request"],
        "schema": {
            "type": "object",
            "properties": {
                "action": {"enum": ["approve", "revise", "request_more_input", "reject"]},
                "update": {"type": "object"},
                "reason": {"type": "string"}
            },
            "required": ["action"]
        }
    })

    action = (v or {}).get("action")
    if action == "approve":
        return Command(goto="b_call", update={"decision": "approved"})

    if action == "reject":
        return Command(
            goto="b_rejected",
            update={"decision": "rejected", "human_note": str(v.get("reason") or "Rejected by human")}
        )

    if action == "request_more_input":
        extra = interrupt({
            "question": "Provide additional constraints before final approval",
            "proposed_request": state["proposed_request"],
            "schema": {
                "type": "object",
                "properties": {
                    "note": {"type": "string"},
                    "update": {"type": "object"}
                }
            }
        })
        upd = (extra or {}).get("update") or {}
        new_req = _merge_request(state["proposed_request"], upd)
        note = str((extra or {}).get("note") or "Additional human input captured")
        return Command(
            goto="b_gate",
            update={
                "proposed_request": new_req,
                "decision": "awaiting_final_approval",
                "human_note": note,
            },
        )

    # revise path (default if action missing/invalid)
    upd = (v or {}).get("update") or {}
    new_req = _merge_request(state["proposed_request"], upd)
    return Command(goto="b_gate", update={"proposed_request": new_req, "decision": "revised"})

def b_call(state: BState) -> BState:
    import requests
    r = requests.get(state["proposed_request"]["url"], params=state["proposed_request"].get("params"), timeout=10)
    return {
        "api_result": {"status_code": r.status_code, "url": r.url},
        "decision": state.get("decision", "approved")
    }

def b_rejected(state: BState) -> BState:
    return {
        "api_result": {"status": "skipped", "reason": state.get("human_note", "Rejected by human")},
        "decision": "rejected",
    }

def build_graph_B():
    g = StateGraph(BState)
    g.add_node("b_propose", b_propose)
    g.add_node("b_gate", b_gate)
    g.add_node("b_call", b_call)
    g.add_node("b_rejected", b_rejected)
    g.set_entry_point("b_propose")
    g.add_edge("b_propose", "b_gate")
    g.add_edge("b_call", END)
    g.add_edge("b_rejected", END)
    return g.compile(checkpointer=CHECKPOINTER)

# Pattern C: Review and edit state

In [7]:
class CState(TypedDict, total=False):
    summary: str

def c_write(state: CState) -> CState:
    text = LLM.invoke("Write 2 sentences about why human in the loop matters for agents").content
    return {"summary": text}

def c_edit(state: CState) -> CState:
    res = interrupt({
        "task": "Edit the summary text",
        "summary": state["summary"],
        "schema": {
            "type": "object",
            "properties": {"edited_text": {"type": "string"}},
            "required": ["edited_text"]
        }
    })
    return {"summary": res["edited_text"]}

def build_graph_C():
    g = StateGraph(CState)
    g.add_node("c_write", c_write)
    g.add_node("c_edit", c_edit)
    g.set_entry_point("c_write")
    g.add_edge("c_write", "c_edit")
    g.add_edge("c_edit", END)
    return g.compile(checkpointer=CHECKPOINTER)

# Pattern D: Parallel interrupts with single resume map

In [8]:
class DState(TypedDict, total=False):
    text_1: str
    text_2: str

def d_h1(state: DState):
    v = interrupt({"text_to_revise": state["text_1"]})
    return {"text_1": v}

def d_h2(state: DState):
    v = interrupt({"text_to_revise": state["text_2"]})
    return {"text_2": v}

def build_graph_D():
    g = StateGraph(DState)
    g.add_node("d_h1", d_h1)
    g.add_node("d_h2", d_h2)
    g.add_edge(START, "d_h1")
    g.add_edge(START, "d_h2")
    g.add_edge("d_h1", END)
    g.add_edge("d_h2", END)
    return g.compile(checkpointer=CHECKPOINTER)

# Pattern E: Tool call review in a tiny ReAct loop

In [9]:
def add_hitl(tool_obj: BaseTool | Any) -> BaseTool:
    if not isinstance(tool_obj, BaseTool):
        tool_obj = tool(tool_obj)

    @tool(tool_obj.name, description=tool_obj.description, args_schema=tool_obj.args_schema)
    def wrapped(**tool_input):
        request = [{
            "action_request": {"action": tool_obj.name, "args": tool_input},
            "config": {"allow_accept": True, "allow_edit": True, "allow_respond": True},
            "description": "Review this tool call"
        }]
        response = interrupt(request)[0]
        if response["type"] == "accept":
            return tool_obj.invoke(tool_input)
        if response["type"] == "edit":
            new_args = response["args"]["args"]
            return tool_obj.invoke(new_args)
        if response["type"] == "response":
            return response["args"]
        raise ValueError("Unsupported interrupt response type")
    return wrapped

@tool("echo_search")
def echo_search(query: str) -> str:
    """Demo tool that simulates a search by echoing the query."""
    return f"Search results for: {query}"

WRAPPED_SEARCH = add_hitl(echo_search)

@task
def e_call_model(messages: List[Dict[str, Any]]):
    return LLM.bind_tools([WRAPPED_SEARCH]).invoke(messages)

@task
def e_call_tool(tool_call: Dict[str, Any]) -> ToolMessage:
    obs = WRAPPED_SEARCH.invoke(tool_call["args"])
    return ToolMessage(content=obs, tool_call_id=tool_call["id"])

from langgraph.func import entrypoint as ep

@ep(checkpointer=CHECKPOINTER)
def e_agent(messages: List[Dict[str, Any]], previous: Optional[List[Dict[str, Any]]] = None):
    if previous is not None:
        messages = add_messages(previous, messages)
    llm_msg = e_call_model(messages).result()
    while True:
        tcs = getattr(llm_msg, "tool_calls", None) or []
        if not tcs:
            break
        tool_results = [e_call_tool(tc).result() for tc in tcs]
        messages = add_messages(messages, [llm_msg, *tool_results])
        llm_msg = e_call_model(messages).result()
    messages = add_messages(messages, llm_msg)
    return ep.final(value=llm_msg, save=messages)

# Pattern F: Static interrupts for debugging

In [10]:
def build_graph_F():
    class S(TypedDict, total=False):
        x: int

    def a(state: S) -> S:
        return {"x": 1}

    def b(state: S) -> S:
        return {"x": state["x"] + 1}

    g = StateGraph(S)
    g.add_node("a", a)
    g.add_node("b", b)
    g.set_entry_point("a")
    g.add_edge("a", "b")
    g.add_edge("b", END)
    return g.compile(
        checkpointer=CHECKPOINTER,
        interrupt_before=["a"],
        interrupt_after=["b"]
    )



# Interactive runner

In [11]:
def wait_for_interrupt_and_prompt(app, cfg):
    """
    Drive interrupts from the terminal.
    Supports single payloads and lists used by wrapped tools.
    Also supports parallel interrupts by auto building a resume map.
    """
    state = app.get_state(cfg)
    ints = getattr(state, "interrupts", []) or []
    if not ints:
        print("No interrupts pending")
        return None

    if len(ints) > 1:
        print("\nMultiple interrupts pending:")
        for i, it in enumerate(ints, 1):
            print(f"[{i}] id={it.interrupt_id} value={jdump(it.value)}")
        print("Enter values per interrupt. Leave blank to echo original.")
        resume_map = {}
        for it in ints:
            val = input(f"Value for {it.interrupt_id}: ").strip()
            if val:
                # Try JSON, else raw string
                try:
                    resume_map[it.interrupt_id] = json.loads(val)
                except Exception:
                    resume_map[it.interrupt_id] = val
            else:
                resume_map[it.interrupt_id] = it.value
        return Command(resume=resume_map)

    # Single interrupt path
    it = ints[0]
    print("\nInterrupt payload:")
    print(jdump(it.value))
    print("Enter resume value. Examples:")
    print("  Pattern B gate: {\"action\": \"approve\"}")
    print("  Pattern B revise: {\"action\": \"revise\", \"update\": {\"params\": {\"limit\": 3}}}")
    print("  Pattern B reject: {\"action\": \"reject\", \"reason\": \"Policy restriction\"}")
    print("  Tool wrapper accept: {\"type\": \"accept\"}")
    print("  Tool wrapper edit: {\"type\": \"edit\", \"args\": {\"args\": {\"query\": \"weather in NY\"}}}")
    print("  Tool wrapper response: {\"type\": \"response\", \"args\": \"Skip tool right now\"}")
    raw = input("resume> ").strip()
    if not raw:
        val = it.value
    else:
        try:
            val = json.loads(raw)
        except Exception:
            val = raw
    return Command(resume=val)

def run_pattern_A():
    app = build_graph_A()
    cfg = {"configurable": {"thread_id": f"A-{uuid.uuid4()}"}}
    topic = input("Enter LinkedIn topic: ").strip() or "Human in the loop for agents"
    stream = app.stream({"linkedin_topic": topic, "human_feedback": []}, config=cfg)
    while True:
        try:
            step = next(stream)
        except StopIteration:
            break
        if "__interrupt__" in step:
            cmd = wait_for_interrupt_and_prompt(app, cfg)
            stream = app.stream(cmd, cfg)
    print("Done A")

def run_pattern_B():
    app = build_graph_B()
    cfg = {"configurable": {"thread_id": f"B-{uuid.uuid4()}"}}
    _ = app.invoke({}, config=cfg)
    while True:
        cmd = wait_for_interrupt_and_prompt(app, cfg)
        _ = app.invoke(cmd, config=cfg)
        state = app.get_state(cfg)
        if not state.interrupts:
            print("Final state:", state.values)
            break
    print("Done B")

def run_pattern_C():
    app = build_graph_C()
    cfg = {"configurable": {"thread_id": f"C-{uuid.uuid4()}"}}
    _ = app.invoke({}, config=cfg)
    cmd = wait_for_interrupt_and_prompt(app, cfg)
    final = app.invoke(cmd, config=cfg)
    print("Final C:", final)

def run_pattern_D():
    app = build_graph_D()
    cfg = {"configurable": {"thread_id": f"D-{uuid.uuid4()}"}}
    _ = app.invoke({"text_1": "alpha", "text_2": "beta"}, config=cfg)
    cmd = wait_for_interrupt_and_prompt(app, cfg)
    final = app.invoke(cmd, config=cfg)
    print("Final D:", final)

def run_pattern_E():
    cfg = {"configurable": {"thread_id": f"E-{uuid.uuid4()}"}}
    user_msg = {"role": "user", "content": "Search for current weather in San Francisco"}
    stream = e_agent.stream([user_msg], cfg)
    while True:
        try:
            step = next(stream)
        except StopIteration:
            break
        if "__interrupt__" in step:
            cmd = wait_for_interrupt_and_prompt(e_agent, cfg)
            stream = e_agent.stream(cmd, cfg)
        else:
            print(step)
    print("Done E")

def run_pattern_F():
    app = build_graph_F()
    cfg = {"configurable": {"thread_id": f"F-{uuid.uuid4()}"}}
    _ = app.invoke({}, config=cfg)
    print("Breakpoint before a recorded. Resuming")
    _ = app.invoke(None, config=cfg)
    print("Breakpoint after b recorded. Resuming")
    final = app.invoke(None, config=cfg)
    print("Final F:", final)

def main():
    menu = """
Pick a demo
1. Human feedback loop for writing
2. HITL checkpoint before sensitive API call
3. Review and edit state
4. Parallel interrupts with resume map
5. Tool call review in a tiny ReAct loop
6. Static interrupts for debugging
q. Quit
> """
    while True:
        choice = input(menu).strip().lower()
        if choice == "1":
            run_pattern_A()
        elif choice == "2":
            run_pattern_B()
        elif choice == "3":
            run_pattern_C()
        elif choice == "4":
            run_pattern_D()
        elif choice == "5":
            run_pattern_E()
        elif choice == "6":
            run_pattern_F()
        elif choice in {"q", "quit", "exit"}:
            sys.exit(0)
        else:
            print("Unknown choice")

if __name__ == "__main__":
    main()



Pick a demo
1. Human feedback loop for writing
2. HITL checkpoint before sensitive API call
3. Review and edit state
4. Parallel interrupts with resume map
5. Tool call review in a tiny ReAct loop
6. Static interrupts for debugging
q. Quit
> 1
Enter LinkedIn topic: write about AI agents

[model] Draft:
AI agents are shifting us from “apps that respond” to “partners that act.”

Instead of clicking through steps, we’ll increasingly set goals—then agents handle the workflow: researching, drafting, scheduling, coordinating tools, and escalating only when needed.

The real differentiator won’t be flashy demos. It’ll be:
- **Reliability** (knowing what to do—and what not to do)
- **Safety & control** (permissions, audit logs, guardrails)
- **Context** (user preferences, business rules, real constraints)
- **Accountability** (clear ownership of outcomes)

What’s your biggest use case for AI agents in your work—and what guardrails would you require before trusting them?


[human] awaiting fee

KeyboardInterrupt: Interrupted by user